# ⚡ Real-time Inference - Water Quality Endpoint

## Purpose
Query the Model Serving endpoint for real-time water quality predictions

## Steps
1. Connect to water quality serving endpoint
2. Test with sample water quality data
3. Demonstrate real-time prediction capabilities
4. Show prediction results and confidence scores

## Notes
- Uses Champion model deployed to serving endpoint
- Real-time predictions for individual water samples
- Standalone notebook (not part of job pipeline)
- Follows Iris MLOps pattern for endpoint testing

In [ ]:
# 📦 Install required packages
%pip install databricks-sdk pandas numpy

# Restart Python 
dbutils.library.restartPython()

In [ ]:
# 📚 Import Libraries
import pandas as pd
import numpy as np
import json
import time
from databricks.sdk import WorkspaceClient
from pyspark.sql import SparkSession

# Initialize
spark = SparkSession.builder.getOrCreate()
w = WorkspaceClient()

print("✅ Libraries loaded for real-time inference testing")

In [ ]:
# 🎯 Configuration
catalog_name = "dev_digital_engineering_services"  # Can be parameterized if needed
schema_name = "default"
env = "dev"  # Or get from parameters

ENDPOINT_NAME = f"water-quality-{env}"

print(f"🎯 Real-time Inference Configuration:")
print(f"   🌐 Endpoint: {ENDPOINT_NAME}")
print(f"   🏷️ Environment: {env}")

In [ ]:
# 🌐 Check Endpoint Status
print("🌐 Checking Model Serving endpoint status...")

try:
    # Get endpoint details
    endpoint = w.serving_endpoints.get(ENDPOINT_NAME)
    
    print(f"✅ Endpoint found: {ENDPOINT_NAME}")
    print(f"📊 Status: {endpoint.state}")
    print(f"🤖 Served Models:")
    
    for model in endpoint.config.served_models:
        print(f"   📦 {model.name}: {model.model_name} (v{model.model_version})")
        print(f"   ⚡ Workload: {model.workload_size}")
        print(f"   🔄 Scale to Zero: {model.scale_to_zero_enabled}")
    
    if str(endpoint.state).lower() != "ready":
        print(f"⚠️ Endpoint status: {endpoint.state}")
        print(f"💡 Endpoint may still be starting up...")
    
except Exception as e:
    print(f"❌ Error accessing endpoint: {e}")
    print(f"💡 Make sure deployment job completed successfully")
    print(f"🔍 Check endpoint '{ENDPOINT_NAME}' in Databricks UI")
    raise e

In [ ]:
# 🧪 Prepare Test Water Quality Samples
print("🧪 Preparing test water quality samples...")

def engineer_features(ph, hardness, solids, chloramines, sulfate, conductivity, 
                     organic_carbon, trihalomethanes, turbidity):
    """Apply feature engineering to raw water quality parameters"""
    
    # Original features
    features = {
        'ph': ph,
        'Hardness': hardness,
        'Solids': solids,
        'Chloramines': chloramines,
        'Sulfate': sulfate,
        'Conductivity': conductivity,
        'Organic_carbon': organic_carbon,
        'Trihalomethanes': trihalomethanes,
        'Turbidity': turbidity
    }
    
    # Engineered features (must match training pipeline)
    features['ph_normalized'] = ph / 14
    features['hardness_ratio'] = hardness / (solids + 1)
    features['organic_load'] = organic_carbon * chloramines
    features['water_quality_index'] = (
        features['ph_normalized'] + 
        (1 / (turbidity + 1)) + 
        (1 / (features['hardness_ratio'] + 1))
    ) / 3
    
    return features

# Test samples representing different water quality scenarios
test_samples = [
    {
        "name": "High Quality Spring Water",
        "params": (7.2, 120, 8000, 3.5, 200, 300, 8, 30, 1.5)
    },
    {
        "name": "Municipal Tap Water", 
        "params": (7.8, 180, 15000, 8.2, 350, 400, 12, 70, 3.5)
    },
    {
        "name": "Hard Well Water",
        "params": (8.1, 320, 25000, 9.8, 420, 580, 18, 95, 5.2)
    },
    {
        "name": "Soft Filtered Water",
        "params": (6.9, 85, 6000, 2.1, 150, 250, 5, 20, 0.8)
    },
    {
        "name": "Questionable Source",
        "params": (5.2, 450, 35000, 15.5, 650, 800, 25, 150, 8.9)
    }
]

print(f"✅ Prepared {len(test_samples)} test samples:")
for sample in test_samples:
    print(f"   🧪 {sample['name']}")

In [ ]:
# ⚡ Run Real-time Predictions
print("⚡ Running real-time predictions...")
print("=" * 60)

results = []

for i, sample in enumerate(test_samples):
    sample_name = sample['name']
    params = sample['params']
    
    print(f"\n🧪 Sample {i+1}: {sample_name}")
    print("-" * 30)
    
    # Apply feature engineering
    features = engineer_features(*params)
    
    # Show key input parameters
    print(f"📊 Input Parameters:")
    print(f"   pH: {features['ph']:.1f}")
    print(f"   Hardness: {features['Hardness']:.1f} mg/L")
    print(f"   Total Solids: {features['Solids']:.0f} mg/L")
    print(f"   Turbidity: {features['Turbidity']:.1f} NTU")
    
    try:
        # Call the endpoint
        start_time = time.time()
        
        response = w.serving_endpoints.query(
            name=ENDPOINT_NAME,
            dataframe_records=[features]
        )
        
        end_time = time.time()
        response_time = (end_time - start_time) * 1000  # Convert to milliseconds
        
        # Parse results
        prediction = response.predictions[0]
        result_text = "✅ SAFE TO DRINK" if prediction == 1 else "❌ NOT SAFE"
        safety_icon = "✅" if prediction == 1 else "⚠️"
        
        print(f"🎯 Prediction: {result_text}")
        print(f"📊 Class: {prediction} (0=Not Potable, 1=Potable)")
        print(f"⏱️ Response Time: {response_time:.1f} ms")
        
        # Store results
        results.append({
            'sample_name': sample_name,
            'prediction': prediction,
            'result_text': result_text,
            'response_time_ms': response_time,
            'ph': features['ph'],
            'hardness': features['Hardness'],
            'turbidity': features['Turbidity']
        })
        
    except Exception as e:
        print(f"❌ Prediction failed: {e}")
        print(f"💡 Endpoint may still be warming up")
        
        # Store error result
        results.append({
            'sample_name': sample_name,
            'prediction': None,
            'result_text': 'ERROR',
            'response_time_ms': None,
            'ph': features['ph'],
            'hardness': features['Hardness'], 
            'turbidity': features['Turbidity']
        })

print(f"\n" + "=" * 60)
print(f"⚡ REAL-TIME INFERENCE TESTING COMPLETED")

In [ ]:
# 📊 Results Summary
print("📊 REAL-TIME INFERENCE RESULTS SUMMARY:")
print("=" * 50)

# Create results DataFrame
results_df = pd.DataFrame(results)

# Filter successful predictions
successful_results = results_df[results_df['prediction'].notna()]

if len(successful_results) > 0:
    print(f"✅ Successful Predictions: {len(successful_results)}/{len(results)}")
    print(f"⏱️ Average Response Time: {successful_results['response_time_ms'].mean():.1f} ms")
    print(f"")
    
    # Show results table
    print("📋 Prediction Results:")
    print("-" * 70)
    print(f"{'Sample':<25} {'pH':>4} {'Hard':>6} {'Turb':>5} {'Result':<15} {'Time':<8}")
    print("-" * 70)
    
    for _, row in successful_results.iterrows():
        result_short = "SAFE" if row['prediction'] == 1 else "UNSAFE" 
        icon = "✅" if row['prediction'] == 1 else "❌"
        
        print(f"{row['sample_name'][:24]:<25} {row['ph']:>4.1f} {row['hardness']:>6.0f} "
              f"{row['turbidity']:>5.1f} {icon} {result_short:<13} {row['response_time_ms']:>5.0f}ms")
    
    # Safety distribution
    safe_count = (successful_results['prediction'] == 1).sum()
    unsafe_count = (successful_results['prediction'] == 0).sum()
    
    print(f"\n📈 Safety Distribution:")
    print(f"   ✅ Safe to Drink: {safe_count} samples")
    print(f"   ❌ Not Safe: {unsafe_count} samples")
    
    # Performance stats
    min_time = successful_results['response_time_ms'].min()
    max_time = successful_results['response_time_ms'].max()
    
    print(f"\n⚡ Performance:")
    print(f"   Fastest: {min_time:.1f} ms")
    print(f"   Slowest: {max_time:.1f} ms") 
    print(f"   Average: {successful_results['response_time_ms'].mean():.1f} ms")
    
else:
    print(f"❌ No successful predictions - endpoint may not be ready")
    print(f"💡 Try again in a few minutes or check endpoint status")

print(f"\n✅ REAL-TIME TESTING COMPLETED!")
print(f"🌐 Endpoint: {ENDPOINT_NAME}")
print(f"🎯 Ready for production real-time water quality predictions")

In [ ]:
# 🔧 Usage Example for Integration
print("🔧 INTEGRATION EXAMPLE:")
print("-" * 30)

print("""
# Example: How to use this endpoint in your applications

from databricks.sdk import WorkspaceClient

# Initialize client
w = WorkspaceClient()

# Prepare water sample data
water_sample = {
    'ph': 7.2, 'Hardness': 180.5, 'Solids': 15000, 
    'Chloramines': 8.2, 'Sulfate': 350, 'Conductivity': 400,
    'Organic_carbon': 12, 'Trihalomethanes': 70, 'Turbidity': 3.5,
    # Engineered features
    'ph_normalized': 7.2/14, 
    'hardness_ratio': 180.5/(15000+1),
    'organic_load': 12*8.2,
    'water_quality_index': (7.2/14 + 1/(3.5+1) + 1/(180.5/(15000+1)+1))/3
}

# Get prediction
response = w.serving_endpoints.query(
    name='water-quality-dev',
    dataframe_records=[water_sample]
)

prediction = response.predictions[0]
result = "SAFE" if prediction == 1 else "UNSAFE"
print(f"Water Quality: {result}")
""")

print(f"🌐 Endpoint URL: Available in Databricks → Machine Learning → Serving")
print(f"📚 API Documentation: Available in endpoint details page")